In [ ]:
import polars as pl
import altair as alt
import pandas #for Datawrapper
import pyarrow
import geopandas as gpd
from shapely.geometry import Point
alt.data_transformers.disable_max_rows()

Graph Ideas:
- Heat Map
- Strip Plot
- Stacked Area
Geospatial:
- Heat map of population by area
- Choropleth of county frequency

In [ ]:
prisoner_df = pl.read_csv("data/prisoner_dataset.csv")
facility_df = pl.read_csv("data/latest_facility_counts.csv")

In [ ]:
#Cleaning/merging

facility_df = facility_df.filter(
    pl.col("State") == "Texas"
).with_columns(
    pl.col("Name").str.split(by=" ").list.first(),
)

facility_df = facility_df.unique(subset=["Name"])

facility_df = facility_df.with_columns(
    pl.col("Name").str.to_titlecase().alias("Current Facility")
)

cutoff_date = pl.lit("09/22/2025").str.to_date("%m/%d/%Y") #From date publish

prisoner_df = prisoner_df.with_columns(
    pl.col("TDCJ Offense").str.to_titlecase(),
    pl.col("Sentence Date").str.to_date("%m/%d/%Y"),
)

prisoner_df = prisoner_df.with_columns(
    (cutoff_date - pl.col("Sentence Date")).dt.total_days().alias("Time Served"),
)

prisoner_df = prisoner_df.with_columns(
    (pl.col("Time Served")/365)
)

breaks = [16, 20, 25, 30, 40, 50, 60, 70, 80]
labels = [
    "<16", "16-20", "20-25", "25-30", "30-40",
    "40-50", "50-60", "60-70", "70-80", "80+"
]

binned_female_prisoners = prisoner_df.with_columns(
    pl.col("Age").cut(breaks, labels=labels).alias("Age_bucket")
)

binned_female_prisoners = binned_female_prisoners.filter(
    (pl.col("Gender") == "F")
)

combined_df = prisoner_df.join(facility_df, on="Current Facility")

race_comparison_df = prisoner_df.filter(
    pl.col("Race").is_in(["W", "H", "B"]),
)

In [ ]:
#10 Most Common Offenses in Texas Jails

def common_offenses(df):
    chart = alt.Chart(df, title="Top 10 Most Common Charges")
    common_bar_chart = chart.mark_bar().encode(
        alt.Y("TDCJ Offense:N").sort("-x"),
        alt.X("count:Q"),
        color = alt.value("#31ce4b")
    ).transform_aggregate(
        count='count()',
        groupby=['TDCJ Offense']
    ).transform_window(
        rank="rank(count)",
        sort=[alt.SortField('count', order='descending')]
    ).transform_filter(
        (alt.datum.rank <= 10)
    )
    return common_bar_chart

common_offenses(prisoner_df)

#data could be cleaned more - see the repeat rows with Aggravated Sexual Assault of a Child

In [ ]:
#Age and average length of sentence

def age_sentence(df):
    chart = alt.Chart(df, title="Average Sentence Length and Time Served by Age")
    age_sentence_chart = chart.mark_line().encode(
        alt.X("Age:Q"),
        alt.Y("average(Sentence (Years)):Q"),
        color=alt.value("#cd1212")
    ).transform_filter(
        (alt.datum["Sentence (Years)"] < 100),
        (alt.datum["Age"] < 100)
    )

    age_served_chart = chart.mark_line().encode(
        alt.X("Age:Q"),
        alt.Y("average(Time Served):Q"),
        color = alt.value("#12cdba")
    ).transform_filter(
        (alt.datum["Age"] < 100)
    )

    final_chart = age_sentence_chart + age_served_chart

    return final_chart

#needs legend

age_sentence(prisoner_df)

In [ ]:
#Scatterchart of prisons by number of prisoners and staff

def prison_pop_scatter(df):
    chart = alt.Chart(df)
    prison_pop_scatter = chart.mark_point().transform_aggregate(
        count="count()",
        staff_confirmed="average(Staff.Confirmed)",
        groupby=["Current Facility"]
    ).encode(
        alt.X("count:Q", title="Prisoner Count"),
        alt.Y("staff_confirmed:Q", title="Staff Confirmed")
    )
    return prison_pop_scatter

prison_pop_scatter(combined_df)

#maybe needs more color?

In [ ]:
def prison_pop_heat(df):
    chart = alt.Chart(df)
    prison_pop_heat = chart.mark_rect().transform_aggregate(
        count="count()",
        staff_confirmed="average(Staff.Confirmed)",
        residents_deaths="average(Residents.Deaths)",
        groupby=["Current Facility"]
    ).encode(
        alt.X("count:Q", title="Prisoner Count"),
        alt.Y("staff_confirmed:Q", title="Staff Confirmed"),
        alt.Color("residents_deaths:Q")
    )

    return prison_pop_heat

prison_pop_heat(combined_df)

In [ ]:
#race on sentence time - stacked area

def race_sentence(df):
    chart = alt.Chart(df)
    race_sentence_chart = chart.mark_area().encode(
        alt.X("Age:Q"),
        alt.Y("average(Sentence (Years)):Q"),
        alt.Color("Race")
    ).transform_filter(
        (alt.datum["Sentence (Years)"] < 100),
        (alt.datum["Age"] < 100)
    )

    return race_sentence_chart

race_sentence(race_comparison_df)

In [ ]:
#race on sentence time - violin plot

def race_sentence_violin(df):
    df = df.filter(
        pl.col('Sentence (Years)').cast(pl.Float64, strict=False).is_not_null()
    )
    chart = alt.Chart(df, width=100).transform_density(
        "Sentence (Years)",
        as_=["Sentence (Years)", "density"],
        extent = [0,100],
        groupby = ["Race"]
    ).mark_area().encode(
        alt.X('density:Q')
            .stack('center')
            .impute(None)
            .title(None)
            .axis(labels=False, values=[0], grid=False, ticks=True),
        alt.Y("Sentence (Years):Q"),
        alt.Color('Race:N'),
        alt.Column('Race:N')
            .spacing(0)
            .header(titleOrient='bottom', labelOrient='bottom', labelPadding=0)
    ).configure_view(
        stroke=None
    )

    return chart

race_sentence_violin(race_comparison_df)

In [ ]:
#Parole review process status versus age histogram?
def parole_review(df):
    chart = alt.Chart(df)
    parole_histogram = chart.mark_bar().encode(
    alt.X("Age:Q", bin=True),
    alt.Y("count():Q"),
    alt.Color("Parole Review Status:N")
    )   
    
    return parole_histogram


parole_review(prisoner_df)

In [ ]:
#radial plot for sentence time for women
binned_female_prisoners = binned_female_prisoners.with_columns(
    pl.col("Age_bucket").cast(pl.String)
)

def women_sentence_times(df):
    df = df.filter(
        pl.col('Sentence (Years)').cast(pl.Float64, strict=False).is_not_null()
    )
    chart = alt.Chart(df)
    base = chart.encode(
        alt.Theta("average(Sentence (Years)):Q").stack(True),
        alt.Radius("Age_bucket:N").scale(type="sqrt", zero=True, rangeMin=20),
        color="Age_bucket:N",
    )

    c1 = base.mark_arc(innerRadius=20, stroke="#fff")

    c2 = base.mark_text(radiusOffset=10).encode(text="average(Sentence (Years)):Q")

    final_radial = c1 + c2
    
    return final_radial

women_sentence_times(binned_female_prisoners)   

In [ ]:
#geospatial of Texas - heatmap of population by area

#AI: I was stuck on where to get started with simple geospatial mapping, so I asked ChatGPT to help me get started. See details in citations.md. 

# use https://altair-viz.github.io/gallery/choropleth.html for counties

geo_combined_df = combined_df.to_pandas()

gdf = gpd.GeoDataFrame(
    geo_combined_df,
    geometry=gpd.points_from_xy(geo_combined_df.Latitude, geo_combined_df.Longitude),
    crs="EPSG:4326"  # WGS84 lat/lon
)

texas = gpd.read_file("data/State_Boundary.shp")

ax = texas.plot(color='white', edgecolor='black', figsize=(8,8))
gdf.plot(ax=ax, markersize=gdf['count()'], color='red', alpha=0.6)